# House Price Predictor

End-to-end ML pipeline on California Housing data. Random Forest vs Gradient Boosting with feature importance and SHAP charts.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import shap
import joblib

# color palette
PURPLE = "#534AB7"
TEAL   = "#1D9E75"
CORAL  = "#D85A30"
AMBER  = "#BA7517"
GRAY   = "#888780"
BG     = "#F9F9F9"

SAVE_DIR = "outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("imports done")

imports done


## 1. Load Data

Using California Housing dataset (built into sklearn). 20k samples, 8 features.

In [ ]:
raw = fetch_california_housing(as_frame=True)
df = raw.frame.copy()

Loading California Housing dataset ...


In [5]:
df.columns = [
    "median_income", "house_age", "avg_rooms", "avg_bedrooms",
    "population", "avg_occupancy", "latitude", "longitude",
    "median_house_value",
]

In [6]:
df["median_house_value"] *= 100_000  # convert to USD

In [7]:
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(df.head())

Rows: 20,640 | Columns: 9
   median_income  house_age  avg_rooms  avg_bedrooms  population  \
0         8.3252       41.0   6.984127      1.023810       322.0   
1         8.3014       21.0   6.238137      0.971880      2401.0   
2         7.2574       52.0   8.288136      1.073446       496.0   
3         5.6431       52.0   5.817352      1.073059       558.0   
4         3.8462       52.0   6.281853      1.081081       565.0   

   avg_occupancy  latitude  longitude  median_house_value  
0       2.555556     37.88    -122.23            452600.0  
1       2.109842     37.86    -122.22            358500.0  
2       2.802260     37.85    -122.24            352100.0  
3       2.547945     37.85    -122.25            341300.0  
4       2.181467     37.85    -122.25            342200.0  


## 2. EDA & Preprocessing

Just a quick look at distributions and correlations. Nothing fancy.

In [12]:
print(df.describe().round(2))

# correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved correlation_heatmap.png")

       median_income  house_age  avg_rooms  avg_bedrooms  population  \
count       20640.00   20640.00   20640.00      20640.00    20640.00   
mean            3.87      28.64       5.43          1.10     1425.48   
std             1.90      12.59       2.47          0.47     1132.46   
min             0.50       1.00       0.85          0.33        3.00   
25%             2.56      18.00       4.44          1.01      787.00   
50%             3.53      29.00       5.23          1.05     1166.00   
75%             4.74      37.00       6.05          1.10     1725.00   
max            15.00      52.00     141.91         34.07    35682.00   

       avg_occupancy  latitude  longitude  median_house_value  
count       20640.00  20640.00   20640.00            20640.00  
mean            3.07     35.63    -119.57           206855.82  
std            10.39      2.14       2.00           115395.62  
min             0.69     32.54    -124.35            14999.00  
25%             2.43     33.93 

## 3. Feature Engineering

Created a few ratio features. Rooms per person, bedrooms per room, income per occupant. These usually help.

In [13]:
df["rooms_per_person"]    = df["avg_rooms"]    / (df["avg_occupancy"] + 1e-6)
df["bedrooms_per_room"]   = df["avg_bedrooms"] / (df["avg_rooms"]    + 1e-6)
df["income_per_occupant"] = df["median_income"]/ (df["avg_occupancy"] + 1e-6)

In [14]:
FEATURES = [
    "median_income", "house_age", "avg_rooms", "avg_bedrooms",
    "population", "avg_occupancy", "latitude", "longitude",
    "rooms_per_person", "bedrooms_per_room", "income_per_occupant",
]

In [15]:
TARGET = "median_house_value"

X = df[FEATURES]
y = df[TARGET]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
print(f"train: {len(X_train):,} | test: {len(X_test):,}")

train: 16,512 | test: 4,128


## 4. Modeling

Training Random Forest and Gradient Boosting. RF with 200 trees, GB with 200 estimators and 0.08 learning rate.

In [18]:
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=4,
    random_state=42, n_jobs=-1
)

In [19]:

gb_model = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.08,
    subsample=0.8, random_state=42
)


In [20]:
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

GradientBoostingRegressor(learning_rate=0.08, max_depth=5, n_estimators=200,
                          random_state=42, subsample=0.8)

## 5. Evaluation

Comparing MAE, RMSE, R2 and 5-fold CV R2.

In [21]:
def evaluate(model, X_tr, X_te, y_tr, y_te, name):
    y_pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    cv_r2 = cross_val_score(model, X_tr, y_tr, cv=5, scoring="r2").mean()
    print(f"\n[{name}]")
    print(f"  MAE  : ${mae:>10,.0f}")
    print(f"  RMSE : ${rmse:>10,.0f}")
    print(f"  R2   :  {r2:.4f}")
    print(f"  CV R2:  {cv_r2:.4f}")
    return {
        "name": name, "MAE": mae, "RMSE": rmse,
        "R2": r2, "CV_R2": cv_r2, "y_pred": y_pred
    }

print("evaluation results:")
rf_res = evaluate(rf_model, X_train, X_test, y_train, y_test, "Random Forest")
gb_res = evaluate(gb_model, X_train, X_test, y_train, y_test, "Gradient Boosting")

evaluation results:

[Random Forest]
  MAE  : $    32,008
  RMSE : $    49,515
  R2   :  0.8129
  CV R2:  0.8100

[Gradient Boosting]
  MAE  : $    31,138
  RMSE : $    47,218
  R2   :  0.8299
  CV R2:  0.8315


## 6. Plots

### Predicted vs Actual

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

In [23]:
colors = [PURPLE, TEAL]
for ax, res, color in zip(axes, [rf_res, gb_res], colors):
    ax.set_facecolor(BG)
    ax.scatter(y_test / 1000, res["y_pred"] / 1000,
               alpha=0.3, s=10, color=color, label="Predictions")
    lim = [y_test.min() / 1000, y_test.max() / 1000]
    ax.plot(lim, lim, "r--", lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Price ($K)", fontsize=11)
    ax.set_ylabel("Predicted Price ($K)", fontsize=11)
    ax.set_title(f"{res['name']}\nR2 = {res['R2']:.3f} | MAE = ${res['MAE']:,.0f}",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

In [24]:
plt.suptitle("Model Comparison: Predicted vs Actual House Prices",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

### Feature Importances

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

In [26]:
for ax, model, res, color in zip(
        axes, [rf_model, gb_model], [rf_res, gb_res], [PURPLE, TEAL]):
    ax.set_facecolor(BG)
    importances = pd.Series(model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)
    bars = ax.barh(importances.index, importances.values, color=color, alpha=0.85)
    ax.set_xlabel("Importance Score", fontsize=11)
    ax.set_title(f"{res['name']}\nFeature Importances", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.2, axis="x")
    for bar, val in zip(bars, importances.values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", ha="left", fontsize=8)


In [27]:
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()

## 7. SHAP Explainability

TreeExplainer on Gradient Boosting. Summary plot + waterfall for a single prediction.

In [28]:
X_sample = X_test.sample(500, random_state=42)

In [29]:
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer(X_sample)

In [30]:
# summary plot
plt.figure(figsize=(10, 7))
plt.gcf().patch.set_facecolor(BG)
shap.summary_plot(shap_values, X_sample, show=False, plot_size=None)
plt.title("SHAP Summary Plot — Feature Impact on Predictions",
          fontsize=13, fontweight="bold", pad=10)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/shap_summary.png", dpi=150, bbox_inches="tight")
plt.close()

In [31]:
# waterfall for single prediction
plt.figure(figsize=(10, 5))
plt.gcf().patch.set_facecolor(BG)
shap.plots.waterfall(shap_values[0], show=False)
plt.title("SHAP Waterfall — Single Prediction Explanation",
          fontsize=13, fontweight="bold", pad=10)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()

## 8. Save Models

In [32]:
joblib.dump(rf_model, f"{SAVE_DIR}/random_forest.pkl")
joblib.dump(gb_model, f"{SAVE_DIR}/gradient_boosting.pkl")
print("models saved to outputs/")

models saved to outputs/


## Summary

Final results comparing both models.

In [33]:
print("="*55)
print("  FINAL RESULTS SUMMARY")
print("="*55)
print(f"  {'Metric':<15} {'Random Forest':>18} {'Gradient Boost':>18}")
print("-"*55)
for metric, key in [("R2 Score", "R2"), ("MAE ($)", "MAE"), ("RMSE ($)", "RMSE"), ("CV R2", "CV_R2")]:
    rf_v = rf_res[key]
    gb_v = gb_res[key]
    fmt = ".4f" if key in ("R2", "CV_R2") else ",.0f"
    rf_s = f"{rf_v:{fmt}}" if key in ("R2","CV_R2") else f"${rf_v:{fmt}}"
    gb_s = f"{gb_v:{fmt}}" if key in ("R2","CV_R2") else f"${gb_v:{fmt}}"
    print(f"  {metric:<15} {rf_s:>18} {gb_s:>18}")
print("="*55)
print("\ndone! all plots saved to ./outputs/")

  FINAL RESULTS SUMMARY
  Metric               Random Forest     Gradient Boost
-------------------------------------------------------
  R2 Score                    0.8129             0.8299
  MAE ($)                    $32,008            $31,138
  RMSE ($)                   $49,515            $47,218
  CV R2                       0.8100             0.8315

done! all plots saved to ./outputs/
